# Agregação dos dados INMET — versão Polars (LazyFrame)

Adaptação do notebook original (pandas) para **Polars**, usando `LazyFrame` do
início ao fim. O objetivo é agregar todos os CSVs do INMET (organizados em
subpastas por ano) sem nunca carregar o dataset inteiro na RAM — importante
porque o resultado final pode passar de **16 GB**.

Principais mudanças em relação à versão pandas:

- **`pl.scan_csv` em vez de `pd.read_csv`**: constrói um plano de execução
  (LazyFrame) em vez de ler os dados imediatamente.
- **Um único `scan_csv` para todos os arquivos**, com `missing_columns="insert"`:
  o INMET mudou o layout das colunas ao longo dos anos (algumas estações/anos
  não têm todas as variáveis); o Polars alinha os esquemas automaticamente,
  preenchendo com `null` onde faltar — o equivalente lazy do
  `pd.concat` com colunas divergentes.
- **`decimal_comma=True`** na leitura e na escrita: os arquivos usam vírgula
  como separador decimal (`23,4`), então não precisamos de um passo extra de
  `str.replace(',', '.')`.
- **Colunas de metadado (`ANO_REFERENCIA`, `ESTACAO`, `CODIGO_WMO`, `UF`,
  `REGIAO`) derivadas via expressões do Polars** a partir do caminho do
  arquivo (`include_file_paths`), em vez de um loop Python por arquivo — isso
  mantém tudo dentro do plano lazy/otimizado.
- **`sink_csv` em vez de `.collect().to_csv(...)`**: grava o resultado em
  streaming, processando um pedaço de cada vez, sem materializar o dataset
  inteiro em memória. É essa troca que permite agregar uma base de dezenas de
  GB numa máquina com poucos GB de RAM.

**Nota sobre encoding:** os CSVs do INMET vêm em `latin-1`/`ISO-8859-1`, mas o
leitor CSV do Polars só entende `utf8`/`utf8-lossy` nativamente. Para não
perder os acentos dos cabeçalhos, convertemos cada arquivo para UTF-8 antes de
escaneá-lo — mas isso é feito **linha a linha** (streaming), então o uso de
memória continua O(1) por arquivo, independente do tamanho total da base.

In [1]:
import polars as pl
import glob
import os

LOCALBASEDADOS = r"./base-INM/"

# Define o diretório atual onde os CSVs estão (pode alterar se necessário)
pasta_origem = LOCALBASEDADOS

# Nome do dataset final
arquivo_resultado = "INMET_dataset_agregado.csv"

# Diretório de cache para as cópias em UTF-8 (pode apagar depois de terminar)
diretorio_cache_utf8 = ".cache_utf8"

# Quantas linhas de metadado existem antes do cabeçalho de cada CSV do INMET
LINHAS_CABECALHO = 8

# Nome temporário da coluna que guarda o caminho de origem de cada linha
COL_CAMINHO_ORIGEM = "__source_file"

## 1. Localizar os arquivos

Mesma lógica do notebook original: busca recursiva por `INMET_*.CSV` /
`INMET_*.csv` dentro das subpastas de ano.

In [2]:
def localizar_arquivos_inmet(diretorio_origem):
    """Busca recursivamente os CSVs do INMET nas subpastas de ano."""
    print(f"Buscando arquivos recursivamente em: {diretorio_origem}")

    padrao_maiusculo = os.path.join(diretorio_origem, "**", "INMET_*.CSV")
    padrao_minusculo = os.path.join(diretorio_origem, "**", "INMET_*.csv")

    arquivos = sorted(
        set(glob.glob(padrao_maiusculo, recursive=True))
        | set(glob.glob(padrao_minusculo, recursive=True))
    )

    print(f"Encontrados {len(arquivos)} arquivo(s).")
    return arquivos

## 2. Converter para UTF-8 (streaming, linha a linha)

Isso preserva corretamente os acentos dos cabeçalhos (ex.: `PRECIPITAÇÃO`,
`RADIAÇÃO`) sem nunca carregar um arquivo inteiro na memória — cada arquivo é
lido e regravado linha por linha. A estrutura de pastas (ano/arquivo) é
espelhada dentro de `diretorio_cache_utf8`, para que a coluna `ANO_REFERENCIA`
continue podendo ser extraída do caminho depois.

Arquivos que falharem na conversão (corrompidos, vazios, etc.) são ignorados
com um aviso — igual ao comportamento do `try/except` por arquivo do notebook
original.

In [3]:
def preparar_arquivos_utf8(
    arquivos,
    diretorio_origem,
    skip_rows=LINHAS_CABECALHO,
    diretorio_cache=diretorio_cache_utf8,
    forcar=False,
):
    """
    Os arquivos do INMET são publicados em Latin-1 (ISO-8859-1). O leitor CSV
    do Polars só entende utf8/utf8-lossy nativamente, então convertemos cada
    arquivo para UTF-8 antes do scan_csv. A conversão é feita linha a linha
    (streaming), então o uso de memória é O(1) por arquivo, não O(tamanho da
    base inteira).
    """
    os.makedirs(diretorio_cache, exist_ok=True)
    convertidos = []

    for arquivo in arquivos:
        relativo = os.path.relpath(arquivo, diretorio_origem)
        destino = os.path.join(diretorio_cache, relativo)
        os.makedirs(os.path.dirname(destino) or ".", exist_ok=True)

        try:
            if forcar or not os.path.exists(destino):
                n_linhas = 0
                with open(arquivo, "r", encoding="latin-1", errors="replace") as origem, \
                     open(destino, "w", encoding="utf-8", newline="") as saida:
                    for linha in origem:
                        saida.write(linha)
                        n_linhas += 1

                if n_linhas <= skip_rows:
                    raise ValueError(
                        f"arquivo tem apenas {n_linhas} linha(s), "
                        f"menos que skip_rows={skip_rows}"
                    )

                print(
                    f"✔ Convertido: {os.path.basename(arquivo)} "
                    f"| Linhas de dados: {n_linhas - skip_rows - 1}"
                )

            convertidos.append(destino)

        except Exception as e:
            print(f"✖ Ignorado (falha na conversão): {os.path.basename(arquivo)} -> {e}")

    return convertidos

## 3. Construir o `LazyFrame` agregado

Um único `pl.scan_csv` sobre a lista inteira de arquivos. Nada é lido de fato
ainda — apenas o plano de execução é montado:

- `missing_columns="insert"` alinha esquemas diferentes entre anos/estações
  (equivalente lazy ao `pd.concat` com colunas divergentes).
- `include_file_paths=COL_CAMINHO_ORIGEM` guarda o caminho de origem de cada
  linha, usado a seguir para extrair `ANO_REFERENCIA`, `ESTACAO`,
  `CODIGO_WMO`, `UF` e `REGIAO` via expressões (`str.split`, `list.get`) —
  tudo dentro do plano lazy, sem loop Python linha a linha.
- Colunas "Unnamed" (a coluna vazia extra que sobra do `;` final de cada linha
  do INMET) são removidas, assim como no notebook original.

In [4]:
def construir_lazyframe_agregado(arquivos, skip_rows=LINHAS_CABECALHO):
    """Monta o LazyFrame agregado de todos os arquivos, sem executar nada ainda."""
    if not arquivos:
        raise FileNotFoundError("Nenhum arquivo INMET válido para processar.")

    lf = pl.scan_csv(
        arquivos,
        separator=";",
        skip_rows=skip_rows,
        encoding="utf8",
        decimal_comma=True,        # "23,4" -> 23.4 direto, sem cast manual
        missing_columns="insert",  # anos com colunas diferentes -> preenche com null
        infer_schema_length=None,  # varre tudo, evita erro de tipo entre arquivos distintos
        include_file_paths=COL_CAMINHO_ORIGEM,
        ignore_errors=True,
    )

    # -- deriva ANO_REFERENCIA/ESTACAO/CODIGO_WMO/UF/REGIAO a partir do caminho, 100% lazy --
    caminho_normalizado = pl.col(COL_CAMINHO_ORIGEM).str.replace_all(r"\\", "/")
    partes_caminho = caminho_normalizado.str.split("/")
    nome_arquivo = partes_caminho.list.get(-1)
    ano_referencia = partes_caminho.list.get(-2)
    partes_nome = nome_arquivo.str.split("_")

    lf = lf.with_columns(
        ano_referencia.alias("ANO_REFERENCIA"),
        partes_nome.list.get(4).alias("ESTACAO"),
        partes_nome.list.get(3).alias("CODIGO_WMO"),
        partes_nome.list.get(2).alias("UF"),
        partes_nome.list.get(1).alias("REGIAO"),
    )

    # remove colunas "Unnamed" (coluna vazia extra por causa do ";" final de cada linha)
    colunas_atuais = lf.collect_schema().names()
    colunas_metadado = ["ANO_REFERENCIA", "ESTACAO", "CODIGO_WMO", "UF", "REGIAO"]

    colunas_manter = [
        c for c in colunas_atuais
        if c != COL_CAMINHO_ORIGEM and c.strip() != "" and not c.lower().startswith("unnamed")
        and c not in colunas_metadado
    ]

    return lf.select(colunas_metadado + colunas_manter)

## 4. Orquestração: localizar → converter → agregar (streaming) → gravar

O `sink_csv` executa o plano lazy em **streaming**, processando e gravando os
dados em lotes — é o que evita carregar os +16 GB inteiros na RAM de uma vez.
Ao final, a contagem de linhas também é feita via `scan_csv` + `.collect()` de
um único `pl.len()`, sem reabrir o arquivo inteiro em memória.

In [5]:
def agregar_dados_inmet_multi_anos(
    diretorio_origem=LOCALBASEDADOS,
    arquivo_saida="dataset_inmet_agregado_multianos.csv",
):
    """
    Lê todos os arquivos CSV do INMET organizados em subpastas por ano e os
    agrega em um único dataset, adicionando a coluna de ano — versão lazy /
    streaming em Polars, apta a lidar com bases maiores que a RAM disponível.
    """
    arquivos = localizar_arquivos_inmet(diretorio_origem)
    if not arquivos:
        print(f"Nenhum arquivo encontrado nas subpastas de: {diretorio_origem}")
        return

    arquivos_utf8 = preparar_arquivos_utf8(arquivos, diretorio_origem)
    if not arquivos_utf8:
        print("Nenhum arquivo válido após a conversão de encoding.")
        return

    lf = construir_lazyframe_agregado(arquivos_utf8)

    print("\nGravando dataset agregado em streaming (sink_csv)...")
    lf.sink_csv(
        arquivo_saida,
        separator=";",
        decimal_comma=True,   # grava de volta com vírgula decimal, como o arquivo original
        include_bom=False,
    )
    print(f"\nDataset agregado salvo com sucesso em: {arquivo_saida}")

    total = (
        pl.scan_csv(arquivo_saida, separator=";", decimal_comma=True)
        .select(pl.len())
        .collect()
        .item()
    )
    print(f"Total de registros: {total}")

In [6]:
agregar_dados_inmet_multi_anos(pasta_origem, arquivo_resultado)

Buscando arquivos recursivamente em: ./base-INM/
Encontrados 10053 arquivo(s).

Gravando dataset agregado em streaming (sink_csv)...

Dataset agregado salvo com sucesso em: INMET_dataset_agregado.csv
Total de registros: 85064088


## (Opcional) Gravar em Parquet em vez de CSV

Para uma base desse tamanho, **Parquet costuma ser uma escolha melhor que
CSV** como formato final: é colunar, comprimido (normalmente 5-10x menor que o
CSV equivalente) e muito mais rápido de reabrir depois (inclusive com
`pl.scan_parquet`, mantendo o fluxo lazy para as próximas análises). Fica como
sugestão — descomente se fizer sentido no seu pipeline.

In [7]:
arquivos = localizar_arquivos_inmet(pasta_origem)
arquivos_utf8 = preparar_arquivos_utf8(arquivos, pasta_origem)
lf = construir_lazyframe_agregado(arquivos_utf8)
lf.sink_parquet("INMET_dataset_agregado.parquet")

Buscando arquivos recursivamente em: ./base-INM/
Encontrados 10053 arquivo(s).
